# Case B — Notebook 2: Demo Streamlit (B0–B6, Hasil Eksperimen, Laporan)

**Cara pakai (sesi interaktif):**
1. Klik **Edit** / buka notebook ini.
2. Panel kanan: Accelerator = **GPU T4**, Internet = **On**.
3. **Add Input**: dataset **Case B: Multi-label Hate Speech Indonesian Twitter** dan output notebook **Case B: Multilabel Hate Speech Indonesian Twitter** (notebook eksperimen).
4. Klik **Run All**. Di bawah sel *tunnel* muncul link `https://....trycloudflare.com` — buka & bagikan ke tim.

Link hanya aktif selama sesi ini menyala.

In [ ]:
import glob, os, shutil, subprocess, zipfile, sys, time, json
WORK = "/kaggle/working/tbp_project"

def find_project():
    """Cari folder proyek di /kaggle/input (zip otomatis diekstrak Kaggle; bila belum, ekstrak sendiri)."""
    hits = glob.glob("/kaggle/input/**/tbp/config.py", recursive=True)
    hits.sort(key=lambda h: "/datasets/" not in h)      # utamakan dataset (kode & laporan terbaru), bukan output notebook
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    zips = glob.glob("/kaggle/input/**/tbp_project.zip", recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall("/kaggle/tmp_extract")
        return "/kaggle/tmp_extract/tbp_project"
    raise FileNotFoundError("Dataset proyek tidak ditemukan. Tambahkan dataset 'Case B: Multi-label Hate Speech Indonesian Twitter' lewat Add Input.")

src = find_project()
print("Sumber proyek:", src)
if not os.path.exists(WORK):
    shutil.copytree(src, WORK)
os.chdir(WORK)
print("Folder kerja:", os.getcwd())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q transformers==5.23.0 peft==0.22.0 accelerate==1.16.0 sentence-transformers==6.3.0 rank-bm25==0.2.2 streamlit==1.67.0 altair==6.3.0 fastapi uvicorn
import torch, transformers, peft
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("transformers", transformers.__version__, "| peft", peft.__version__)


In [ ]:
# Ambil hasil run (notebook 1) dari Input: tbp_output.zip atau folder hasil
cands = glob.glob("/kaggle/input/**/tbp_output.zip", recursive=True)
if cands:
    zipfile.ZipFile(cands[0]).extractall(WORK)
    print("Hasil dipulihkan dari", cands[0])
else:
    res = glob.glob("/kaggle/input/**/tbp_project/results/results.json", recursive=True)
    if not res:
        raise FileNotFoundError("Output notebook 'Case B: Multilabel Hate Speech Indonesian Twitter' belum ditambahkan lewat Add Input.")
    base = os.path.dirname(os.path.dirname(res[0]))
    for d in ["results", "adapters", "data/processed", "logs"]:
        if os.path.exists(f"{base}/{d}"):
            shutil.copytree(f"{base}/{d}", d, dirs_exist_ok=True)
    print("Hasil dipulihkan dari", base)
print("results.json ada:", os.path.exists("results/results.json"), "| adapter:", os.path.exists("adapters/lora_b5/adapter_config.json"))


In [ ]:
# Nyalakan API (FastAPI :8000) dan UI Streamlit (:8501) di background
os.makedirs("logs", exist_ok=True)
env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
api = subprocess.Popen([sys.executable, "-m", "uvicorn", "tbp.serve:app", "--port", "8000"],
                       stdout=open("logs/serve_demo.log", "w"), stderr=subprocess.STDOUT, env=env)
import urllib.request
def up(url):
    try:
        urllib.request.urlopen(url, timeout=2); return True
    except Exception:
        return False
for _ in range(120):
    if up("http://localhost:8000/health"): break
    time.sleep(2)
print("API siap:", up("http://localhost:8000/health"))
ui = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app_streamlit.py", "--server.port", "8501",
                       "--server.headless", "true"], stdout=open("logs/streamlit.log", "w"), stderr=subprocess.STDOUT, env=env)
for _ in range(60):
    if up("http://localhost:8501/_stcore/health"): break
    time.sleep(2)
print("Streamlit siap:", up("http://localhost:8501/_stcore/health"))


In [ ]:
# Link publik sementara via Cloudflare Quick Tunnel (tanpa akun). Link berubah setiap sesi.
import re, stat, urllib.request
cf = "/kaggle/working/cloudflared"
if not os.path.exists(cf):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    os.chmod(cf, os.stat(cf).st_mode | stat.S_IEXEC)
tun = subprocess.Popen([cf, "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 90 and not url:
    line = tun.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
print("=" * 70)
print("BUKA DEMO DI:", url or "(gagal membuat tunnel - cek Internet ON)")
print("Bagikan link ini ke anggota tim. Link aktif selama sesi notebook ini menyala.")
print("=" * 70)


In [ ]:
# Mode "Interactive" (dibuka di browser): sel ini menjaga sesi tetap aktif selama demo (hentikan dengan tombol Stop).
# Klasifikasi pertama butuh ±30 detik (memuat model), berikutnya ±15–40 detik untuk B0–B6 sekaligus.
# Mode "Batch" (Save & Run All / API): lakukan uji otomatis lalu selesai, agar notebook tidak menyala 12 jam.
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Batch":
    req = urllib.request.Request("http://localhost:8000/compare", data=json.dumps({"text": "USER dasar goblok lu, cebong semua sama aja"}).encode(),
                                 headers={"Content-Type": "application/json"})
    rows = json.loads(urllib.request.urlopen(req, timeout=900).read())
    for r in rows:
        print(r["config"], "ABSTAIN" if r["abstain"] else r["labels"], r["evidence_spans"], f"{r['latency_ms']:.0f} ms")
    try:
        page = urllib.request.urlopen(url, timeout=60).status if url else None
    except Exception as e:
        page = f"error: {e}"
    print("SELF-TEST:", "OK" if len(rows) == 7 and up("http://localhost:8501/_stcore/health") and page == 200 else "PERLU DICEK",
          "| tunnel HTTP", page)
    for p_ in (tun, ui, api):
        p_.terminate()
else:
    try:
        while True:
            time.sleep(60)
            print(time.strftime("%H:%M"), "| API", api.poll() is None, "| UI", ui.poll() is None, "| tunnel", tun.poll() is None, flush=True)
    except KeyboardInterrupt:
        print("Dihentikan.")
